# Stage 3 Instruction Training

Train the profile-to-protein instruction model from the configured instruction JSONL stream. The implementation lives in `libs/instruction`; reference folders stay read-only. The trainer prints icon-labelled, detailed logs for setup, data counts, training steps, eval, checkpoints, plots, and summaries.

Supports Windows/Jupyter, Ubuntu/Linux cloud VMs, and Google Colab.

Default config: `config/instruction.16gb.yaml`. Use `MDNAC_INSTRUCTION_CONFIG` only when you want another instruction YAML. Use `MDNAC_REPO_DIR` only when the notebook is not opened from inside the cloned repo.


In [1]:
import json
import os
import platform
import sys
from pathlib import Path

import torch

def find_repo_dir_for_import(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        candidates.append(Path(repo_dir_env).expanduser())
    candidates.extend([
        Path("/content/MDNAC"),
        Path("/content/drive/MyDrive/MDNAC"),
    ])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError(
        "Could not locate repo. Run inside the repo, set MDNAC_REPO_DIR, "
        "or in Colab clone/mount the repo under /content or Google Drive."
    )


REPO_DIR = find_repo_dir_for_import(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.notebook_runtime import apply_instruction_notebook_overrides, bootstrap_notebook
from libs.core.pretrain.training_config import _load_yaml_mapping
from libs.instruction import (
    InstructionTrainer,
    audit_instruction_jsonl,
    build_instruction_training_config,
)

RUNTIME = bootstrap_notebook(REPO_DIR)
REPO_DIR = Path(RUNTIME["repo_dir"])
print(json.dumps(RUNTIME, indent=2, ensure_ascii=False))


{
  "repo_dir": "C:\\Users\\Admin\\Desktop\\MDNAC",
  "platform": "Windows",
  "platform_name": "Windows",
  "is_colab": false,
  "is_notebook": true,
  "python": "3.11.15",
  "cuda_available": true,
  "cuda_device_count": 1
}


In [2]:
CONFIG_PATH = Path(os.environ.get("MDNAC_INSTRUCTION_CONFIG", REPO_DIR / "config" / "instruction.16gb.yaml")).expanduser()
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = (REPO_DIR / CONFIG_PATH).resolve()
raw_config = _load_yaml_mapping(CONFIG_PATH)
CONFIG_BASE = build_instruction_training_config(REPO_DIR, raw_config)
CONFIG = apply_instruction_notebook_overrides(CONFIG_BASE)
notebook_overrides = []
if CONFIG.num_workers != CONFIG_BASE.num_workers:
    notebook_overrides.append(f"data.num_workers: {CONFIG_BASE.num_workers} -> {CONFIG.num_workers}")
if CONFIG.multi_gpu_mode != CONFIG_BASE.multi_gpu_mode:
    notebook_overrides.append(f"runtime.multi_gpu_mode: {CONFIG_BASE.multi_gpu_mode} -> {CONFIG.multi_gpu_mode}")

config_preview = {
    "config_path": str(CONFIG_PATH),
    "notebook_overrides": notebook_overrides,
    "instruction_jsonl_paths": [str(path) for path in CONFIG.instruction_jsonl],
    "artifact_source_jsonl": str(CONFIG.artifact_source_jsonl),
    "base_checkpoint_path": str(CONFIG.base_checkpoint_path),
    "output_dir": str(CONFIG.output_dir),
    "artifact_dir": str(CONFIG.artifact_dir),
    "optimizer": {
        "type": CONFIG.optimizer_type,
        "learning_rate": CONFIG.learning_rate,
        "muon_learning_rate": CONFIG.muon_learning_rate,
        "weight_decay": CONFIG.weight_decay,
        "fused": CONFIG.fused,
    },
    "runtime": {
        "platform": platform.system(),
        "platform_name": RUNTIME["platform_name"],
        "is_colab": RUNTIME["is_colab"],
        "running_in_notebook": RUNTIME["is_notebook"],
        "cuda_device_count": torch.cuda.device_count(),
        "device": CONFIG.device,
        "multi_gpu_mode": CONFIG.multi_gpu_mode,
        "mixed_precision": CONFIG.mixed_precision,
    },
    "data": {
        "batch_size": CONFIG.batch_size,
        "gradient_accumulation_steps": CONFIG.gradient_accumulation_steps,
        "log_every_steps": CONFIG.log_every_steps,
        "num_workers": CONFIG.num_workers,
        "shuffle_buffer_size": CONFIG.shuffle_buffer_size,
    },
    "artifacts": {
        "profile_vocab_size": CONFIG.profile_vocab_size,
        "profile_sample_size": CONFIG.artifact_profile_sample_size,
        "sequence_tokenizer_map_path": str(CONFIG.sequence_tokenizer_map_path),
    },
}
print(json.dumps(config_preview, indent=2, ensure_ascii=False))


{
  "config_path": "C:\\Users\\Admin\\Desktop\\MDNAC\\config\\instruction.16gb.yaml",
  "notebook_overrides": [
    "data.num_workers: 4 -> 0"
  ],
  "instruction_jsonl_paths": [
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_1.jsonl",
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_2.jsonl",
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_3.jsonl",
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_4.jsonl",
    "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_5.jsonl"
  ],
  "artifact_source_jsonl": "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\compiled\\refseq_bacteria_protein\\instruction.jsonl",
  "base_checkpoint_path": "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\checkpoints\\protein_from_scratch\\checkpoint_best.pt",
  "output_dir": "C:\\Users\\Admin\\Desktop\\MDNAC\\

In [3]:
def project_path(value) -> Path:
    path = Path(str(value)).expanduser()
    return path if path.is_absolute() else (REPO_DIR / path).resolve()


paths_config = raw_config.get("paths", {})
data_config = raw_config.get("data", {})
DATA_DIR = REPO_DIR / "data"
INSTRUCTION_JSONL = project_path(paths_config.get("instruction_jsonl_path", "data/compiled/refseq_bacteria_protein/instruction.jsonl"))
INSTRUCTION_PART_DIR = project_path(paths_config.get("instruction_part_dir", INSTRUCTION_JSONL.parent))
INSTRUCTION_PART_GLOB = str(data_config.get("instruction_part_glob") or "instruction_part_*.jsonl")
BASE_CHECKPOINT = Path(CONFIG.base_checkpoint_path)
OUTPUT_DIR = Path(CONFIG.output_dir)
ARTIFACT_DIR = Path(CONFIG.artifact_dir)
ARTIFACT_SOURCE_JSONL = Path(CONFIG.artifact_source_jsonl)

for folder in (DATA_DIR, INSTRUCTION_JSONL.parent, INSTRUCTION_PART_DIR, OUTPUT_DIR, ARTIFACT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

instruction_part_paths = sorted(INSTRUCTION_PART_DIR.glob(INSTRUCTION_PART_GLOB))
data_setup = {
    "data_dir": str(DATA_DIR),
    "compiled_instruction_dir": str(INSTRUCTION_JSONL.parent),
    "instruction_jsonl": {
        "path": str(INSTRUCTION_JSONL),
        "exists": INSTRUCTION_JSONL.exists(),
    },
    "instruction_parts": {
        "dir": str(INSTRUCTION_PART_DIR),
        "glob": INSTRUCTION_PART_GLOB,
        "count": len(instruction_part_paths),
        "preview": [str(path) for path in instruction_part_paths[:5]],
    },
    "artifact_source_jsonl": {
        "path": str(ARTIFACT_SOURCE_JSONL),
        "exists": ARTIFACT_SOURCE_JSONL.exists(),
    },
    "base_checkpoint": {
        "path": str(BASE_CHECKPOINT),
        "exists": BASE_CHECKPOINT.exists(),
    },
    "output_dir": str(OUTPUT_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
}
print(json.dumps(data_setup, indent=2, ensure_ascii=False))

{
  "data_dir": "C:\\Users\\Admin\\Desktop\\MDNAC\\data",
  "compiled_instruction_dir": "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\compiled\\refseq_bacteria_protein",
  "instruction_jsonl": {
    "path": "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\compiled\\refseq_bacteria_protein\\instruction.jsonl",
    "exists": false
  },
  "instruction_parts": {
    "dir": "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts",
    "glob": "instruction_part_*.jsonl",
    "count": 5,
    "preview": [
      "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_1.jsonl",
      "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_2.jsonl",
      "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_3.jsonl",
      "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_train_parts\\instruction_part_4.jsonl",
      "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\cache\\instruction_trai

In [4]:
audit = audit_instruction_jsonl(
    CONFIG.instruction_jsonl,
    default_sequence_type=CONFIG.default_sequence_type,
    instruction_field=CONFIG.instruction_field,
    input_field=CONFIG.input_field,
    output_field=CONFIG.output_field,
)
print(json.dumps(audit.to_dict(), indent=2, ensure_ascii=False))

KeyboardInterrupt: 

In [4]:
trainer = InstructionTrainer(CONFIG)
result = trainer.train()
print(json.dumps(result.to_dict(), indent=2, ensure_ascii=False))

🚀 Starting profile-to-protein instruction training
🧾 Instruction JSONL files: 5
   📄 [1/5] C:\Users\Admin\Desktop\MDNAC\data\cache\instruction_train_parts\instruction_part_1.jsonl
   📄 [2/5] C:\Users\Admin\Desktop\MDNAC\data\cache\instruction_train_parts\instruction_part_2.jsonl
   📄 [3/5] C:\Users\Admin\Desktop\MDNAC\data\cache\instruction_train_parts\instruction_part_3.jsonl
   📄 [4/5] C:\Users\Admin\Desktop\MDNAC\data\cache\instruction_train_parts\instruction_part_4.jsonl
   📄 [5/5] C:\Users\Admin\Desktop\MDNAC\data\cache\instruction_train_parts\instruction_part_5.jsonl
📥 Base checkpoint: C:\Users\Admin\Desktop\MDNAC\data\checkpoints\protein_from_scratch\checkpoint_best.pt


📁 Output dir: C:\Users\Admin\Desktop\MDNAC\data\checkpoints\protein_instruction
🧬 Artifact dir: C:\Users\Admin\Desktop\MDNAC\data\compiled\refseq_bacteria_instruction_profile
⚙️  Training config: epochs=3 | batch_size=2 | grad_accum=16 | eval_freq=20 | eval_batches=64 | max_steps=None
📦 [1/7] Preparing profile-aware instruction artifacts...
   ✅ Reusing tokenizer map: C:\Users\Admin\Desktop\MDNAC\data\compiled\refseq_bacteria_instruction_profile\tokenizer_map.json
✅ Artifacts ready | vocab_size=513 | records=100,000 | sequence_type=protein
🧠 [2/7] Building model from base checkpoint...
   📥 Loading checkpoint payload: C:\Users\Admin\Desktop\MDNAC\data\checkpoints\protein_from_scratch\checkpoint_best.pt
The MDC fast path is unavailable (missing optional libraries: causal-conv1d, flash-linear-attention). Falling back to the torch implementation. The fallback uses fp32 computation (2x VRAM for activations).
   🧩 Loading protein backbone for instruction tuning.
✅ Model ready | params=280,5

KeyboardInterrupt: 